# Monitoring, Drift and Observability

Notebook for **Chapter 10** of the *mlops* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/mlops/ch10.html

Tip: change a number, predict the output, then run the cell again.

## 10.1 What to Monitor, and Prometheus and Grafana

In [ ]:
# not run here
from prometheus_client import Counter, Histogram, start_http_server
REQS = Counter("predict_requests_total", "Predict requests", ["model_version"])
LAT = Histogram("predict_latency_seconds", "Latency", buckets=[.01, .025, .05, .1, .25])
start_http_server(9100)
with LAT.time():
    ...  # model call
REQS.labels(model_version="v1").inc()

## 10.2 Data Drift: PSI, KS, Wasserstein and Friends

In [ ]:
def psi(ref, cur, bins=10):
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref); c = np.histogram(cur, edges)[0] / len(cur)
    r, c = np.clip(r, 1e-4, None), np.clip(c, 1e-4, None)   # avoid log(0)
    return float(np.sum((c - r) * np.log(c / r)))

## 10.3 Concept Drift, Prediction Drift and Delayed Labels

## 10.4 Alert Design, Runbooks and Incident Response

In [ ]:
run = 0
for d, p in enumerate(psis, 1):
    run = run + 1 if p > .2 else 0
    print(f"day {d} psi {p:.2f} streak {run}" + ("  ALERT" if run >= 3 else ""))

day 1 psi 0.04 streak 0
day 2 psi 0.05 streak 0
day 3 psi 0.22 streak 1
day 4 psi 0.06 streak 0
day 5 psi 0.25 streak 1
day 6 psi 0.27 streak 2
day 7 psi 0.31 streak 3  ALERT
day 8 psi 0.30 streak 4  ALERT


## 10.5 Advanced Monitoring: Joint Drift, Label-free Estimates, Slices and Burn Rates

Section 10.2 and 10.3 gave you the single-feature drift tests and the delayed-label problem. This section covers what experts add: checks that see several features together, estimates of performance when labels are late, slice monitoring that does not cry wolf, and alerting and retraining rules driven by numbers. Every output below was produced by code run here. Services such as Prometheus are not needed for these ideas, so nothing here is unrunnable.

In [ ]:
import numpy as np
from scipy.stats import ks_2samp
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
n = 4000
cov_ref = [[1, 0.8], [0.8, 1]]               # reference: the two features move together
cov_new = [[1, -0.8], [-0.8, 1]]             # today: they move in opposite directions
ref = rng.multivariate_normal([0, 0], cov_ref, n)
new = rng.multivariate_normal([0, 0], cov_new, n)

for j in range(2):                           # one feature at a time: nothing to see
    print(f"feature {j}: KS p-value {ks_2samp(ref[:, j], new[:, j]).pvalue:.2f}")

# Domain classifier: can a model tell reference rows from new rows?
X = np.vstack([ref, new]); y = np.r_[np.zeros(n), np.ones(n)]
p = cross_val_predict(RandomForestClassifier(100, min_samples_leaf=20, random_state=0, n_jobs=1), X, y, cv=5, method="predict_proba")[:, 1]
print("domain classifier AUC:", round(roc_auc_score(y, p), 3), "(0.5 = cannot tell apart)")

feature 0: KS p-value 0.50
feature 1: KS p-value 0.25
domain classifier AUC: 0.895 (0.5 = cannot tell apart)


In [ ]:
import numpy as np
# MMD (maximum mean discrepancy) with an RBF kernel: one number for "how different are two clouds of points?"
def mmd2(a, b, gamma=0.5):
    def k(x, y):
        d = ((x[:, None, :] - y[None, :, :]) ** 2).sum(-1)
        return np.exp(-gamma * d)
    return k(a, a).mean() + k(b, b).mean() - 2 * k(a, b).mean()

rng = np.random.default_rng(1)
base = rng.multivariate_normal([0, 0], [[1, .8], [.8, 1]], 800)
same = rng.multivariate_normal([0, 0], [[1, .8], [.8, 1]], 800)
flip = rng.multivariate_normal([0, 0], [[1, -.8], [-.8, 1]], 800)
shift = rng.multivariate_normal([0.5, 0], [[1, .8], [.8, 1]], 800)
print("same distribution :", round(mmd2(base, same), 4))
print("correlation flipped:", round(mmd2(base, flip), 4))
print("mean shifted 0.5   :", round(mmd2(base, shift), 4))

same distribution : 0.003
correlation flipped: 0.1185
mean shifted 0.5   : 0.0387


In [ ]:
import numpy as np
# Performance estimation without labels (the idea behind CBPE): if the model's probabilities are
# calibrated, expected accuracy = mean of max(p, 1-p) over today's rows. No labels needed.
rng = np.random.default_rng(3)
def day(shift, slope=2.0, n=5000):
    x = rng.normal(shift, 1, n)
    p_true = 1 / (1 + np.exp(-slope * x))          # the real world today
    y = (rng.random(n) < p_true).astype(int)
    return x, y
def model_p(x):                                    # the model learned slope 2 and still believes it
    return 1 / (1 + np.exp(-2 * x))
print("case                      true accuracy  estimated (no labels)")
for name, shift, slope in (("no change", 0.0, 2.0), ("inputs shift +0.5", 0.5, 2.0), ("inputs shift +1.0", 1.0, 2.0),
                           ("concept drift (slope 0.5)", 0.0, 0.5)):
    x, y = day(shift, slope)
    p = model_p(x)
    true_acc = ((p > 0.5) == y).mean()
    est = np.maximum(p, 1 - p).mean()
    print(f"{name:25s} {true_acc:9.3f}      {est:9.3f}")

case                      true accuracy  estimated (no labels)
no change                     0.782          0.781
inputs shift +0.5             0.796          0.796
inputs shift +1.0             0.845          0.845
concept drift (slope 0.5)     0.597          0.779


In [ ]:
import numpy as np
# Monitoring slices: an overall metric hides a failing group, and small groups are noisy.
rng = np.random.default_rng(5)
groups = {"web": (60000, 0.95), "ios": (30000, 0.95), "android": (9000, 0.94), "kiosk": (400, 0.80)}
print("group     n      accuracy   95% interval          alert?")
tot_c = tot_n = 0
for g, (n, acc) in groups.items():
    correct = rng.binomial(n, acc); tot_c += correct; tot_n += n
    p = correct / n; se = (p * (1 - p) / n) ** 0.5
    lo, hi = p - 1.96 * se, p + 1.96 * se
    print(f"{g:8s} {n:6d}   {p:.3f}     [{lo:.3f}, {hi:.3f}]    {'YES' if hi < 0.90 else 'no'}")
print("overall:", round(tot_c / tot_n, 3))

group     n      accuracy   95% interval          alert?
web       60000   0.949     [0.947, 0.951]    no
ios       30000   0.951     [0.948, 0.953]    no
android    9000   0.937     [0.932, 0.942]    no
kiosk       400   0.830     [0.793, 0.867]    YES
overall: 0.948


In [ ]:
# Multi-window burn-rate alerts for an SLO of 99.9% (error budget 0.1%).
# burn rate = observed error rate / allowed error rate. 30 days = 720 hours.
slo_err = 0.001
def burn(err_rate): return err_rate / slo_err
for err in (0.002, 0.0144, 0.05):
    b = burn(err)
    hours_to_empty = 720 / b
    print(f"error rate {err:.2%}: burn rate {b:5.1f}x, budget gone in {hours_to_empty:6.1f} hours")
# Typical rule: page when BOTH the 1-hour and the 5-minute burn rate exceed 14.4
# (14.4x burns 2% of the 30-day budget in one hour).
print("share of monthly budget used in 1 hour at 14.4x:", f"{14.4 / 720:.1%}")

error rate 0.20%: burn rate   2.0x, budget gone in  360.0 hours
error rate 1.44%: burn rate  14.4x, budget gone in   50.0 hours
error rate 5.00%: burn rate  50.0x, budget gone in   14.4 hours
share of monthly budget used in 1 hour at 14.4x: 2.0%


In [ ]:
# When to retrain: compare the expected loss from waiting with the cost of a retrain.
cost_retrain = 1_800            # dollars: compute + checks + people time
decisions_per_day = 20_000
loss_per_error = 4.0            # dollars lost per wrong decision
acc_now, acc_fresh = 0.905, 0.925
daily_loss = decisions_per_day * (acc_fresh - acc_now) * loss_per_error
print("extra loss per day from the stale model: $", round(daily_loss))
print("retrain pays back in", round(cost_retrain / daily_loss, 1), "days")
# Accuracy decay of 0.0015 per day since the last retrain: best interval?
import math
decay = 0.0015                  # accuracy lost per day
# average extra loss over an interval T is decisions * loss * decay * T/2 per day; plus cost/T per day
best = math.sqrt(2 * cost_retrain / (decisions_per_day * loss_per_error * decay))
print("best retrain interval:", round(best, 1), "days")

extra loss per day from the stale model: $ 1600
retrain pays back in 1.1 days
best retrain interval: 5.5 days


## Chapter recap

## Practice set

**1.** (Easy) A counter reads 1,200 at 10:00:00 and 1,500 at 10:01:00. What is the rate per second?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/mlops/ch10.html